# Notebook 08 — Final Analysis & Publication Figures

Day 10–11 goals:
1. Top-10 candidate refinement (exhaustiveness=32)
2. All publication-quality figures (Fig 1–5)
3. Radar chart: generated vs baselines
4. 3D pose visualization of Top-10
5. Full summary table for report


In [ ]:
import os, sys, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src import load_config, VinaDocker
from src.visualization import plot_score_distribution, plot_chemical_space, plot_radar, render_3d_pose
from src.utils import merge_vina_and_metrics, filter_top_candidates
import pandas as pd

cfg = load_config(f'{PROJECT_ROOT}/configs/targets.yaml')
with open(f'{PROJECT_ROOT}/data/pocket_centers.json') as f:
    centers = json.load(f)

In [ ]:
# ── Load all results ─────────────────────────────────────────────────────────
wt_combined = merge_vina_and_metrics(
    f'{PROJECT_ROOT}/results/vina_scores/1M17_vina.csv',
    f'{PROJECT_ROOT}/results/vina_scores/1M17_metrics.csv',
)
mut_combined = merge_vina_and_metrics(
    f'{PROJECT_ROOT}/results/vina_scores/4I22_vina.csv',
    f'{PROJECT_ROOT}/results/vina_scores/4I22_metrics.csv',
)
baselines_df = pd.read_csv(f'{PROJECT_ROOT}/results/vina_scores/baselines.csv')

print('WT molecules:', len(wt_combined))
print('T790M molecules:', len(mut_combined))

In [ ]:
# ── Top-10 WT with refined docking (exhaustiveness=32) ───────────────────────
top10_wt = filter_top_candidates(wt_combined, top_k=10)

docker_refined = VinaDocker(
    receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/1M17_receptor.pdbqt',
    center=tuple(centers['1M17']),
    exhaustiveness=32,
)
from rdkit import Chem

refined_scores = []
for _, row in top10_wt.iterrows():
    mol = Chem.MolFromSmiles(row['smiles'])
    if mol:
        score = docker_refined.dock_mol(mol)
        refined_scores.append(score)
    else:
        refined_scores.append(None)

top10_wt = top10_wt.copy()
top10_wt['vina_refined'] = refined_scores
top10_wt.to_csv(f'{PROJECT_ROOT}/results/vina_scores/1M17_top10_refined.csv', index=False)
top10_wt[['smiles', 'vina_score', 'vina_refined', 'qed', 'sa_score', 'MW', 'LogP']]

In [ ]:
# ── Figure 3: Radar chart ────────────────────────────────────────────────────
# Build a baseline_df with merged scores
base_smiles = {n: info['smiles'] for n, info in cfg['baselines'].items()}
from src.evaluation import MolEvaluator
evaluator = MolEvaluator()
base_metrics = evaluator.evaluate(list(base_smiles.values()))
base_vina = baselines_df[baselines_df['target'] == '1M17'][['drug', 'vina_score']].set_index('drug')
base_metrics['vina_score'] = base_metrics.index.map(lambda i: base_vina['vina_score'].iloc[i] if i < len(base_vina) else None)

plot_radar(top10_wt, base_metrics,
           output_path=f'{PROJECT_ROOT}/results/figures/fig3_radar.png')

In [ ]:
# ── 3D pose visualization of Top-10 ─────────────────────────────────────────
from IPython.display import display

for i, row in top10_wt.head(5).iterrows():
    print(f'--- Candidate {i+1}  Vina={row["vina_score"]:.2f}  QED={row["qed"]:.3f} ---')
    view = render_3d_pose(row['smiles'], title=f'Candidate {i+1}')
    if view:
        display(view)

In [ ]:
# ── Summary statistics table for report ─────────────────────────────────────
from src.evaluation import MolEvaluator
from src.utils import sdf_to_smiles_list

ev = MolEvaluator()
summary = []
for label, sdf, vina_csv in [
    ('EGFR WT (1M17)', f'{PROJECT_ROOT}/results/generated/1M17_raw.sdf',
     f'{PROJECT_ROOT}/results/vina_scores/1M17_vina.csv'),
    ('EGFR T790M (4I22)', f'{PROJECT_ROOT}/results/generated/4I22_raw.sdf',
     f'{PROJECT_ROOT}/results/vina_scores/4I22_vina.csv'),
]:
    smiles = sdf_to_smiles_list(sdf)
    df = ev.evaluate(smiles)
    vina = pd.read_csv(vina_csv)
    summary.append({
        'Target': label,
        'Total Generated': len(smiles),
        'Validity': f"{ev.validity(df):.1%}",
        'Uniqueness': f"{ev.uniqueness(df):.1%}",
        'Diversity': f"{ev.diversity(df):.3f}",
        'Mean QED': f"{df['qed'].mean():.3f}",
        'Mean SA': f"{df['sa_score'].mean():.2f}",
        'Lipinski Pass': f"{df['Lipinski'].mean():.1%}",
        'Mean Vina': f"{vina['vina_score'].mean():.2f}",
        'Best Vina': f"{vina['vina_score'].min():.2f}",
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(f'{PROJECT_ROOT}/results/summary_table.csv', index=False)
summary_df